<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/Vectorized_Multi-Layer_Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
# Fix the random seed so the initialized weights remain the same
np.random.seed(42)

In [3]:
# ReLU activation function used in hidden layers.
# It introduces non-linearity, allowing the network
# to learn complex patterns.
def relu(x):
    return np.maximum(0, x)

# Derivative of ReLU required during backpropagation.
# Gradients flow only through neurons that were activated.
def relu_derivative(x):
    return (x > 0).astype(float)

# Softmax activation converts output logits into probability values for multi-class classification.
def softmax(z):

    # Numerical stability trick to avoid exponential overflow.
    z = z - np.max(z, axis=1, keepdims=True)

    exp = np.exp(z)

    # Normalize the exponential values into probabilities.
    return exp / np.sum(exp, axis=1, keepdims=True)

In [4]:
# Cross-Entropy measures the difference between
# predicted probabilities and true class labels.
# Lower loss indicates better model predictions.
def cross_entropy_loss(y_true, y_pred):

    m = y_true.shape[0]

    # Prevent logarithm of zero.
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

    loss = -np.sum(y_true * np.log(y_pred)) / m

    return loss

In [5]:
# Convert integer class labels into one-hot vectors.
def one_hot(y, num_classes):

    onehot = np.zeros((len(y), num_classes))

    onehot[np.arange(len(y)), y] = 1

    return onehot

In [6]:
# Initialize weights and biases for every layer.
# He Initialization improves convergence for ReLU networks.
def initialize_parameters(layer_dims):
    parameters = {}
    for l in range(1, len(layer_dims)):
        parameters["W"+str(l)] = np.random.randn(
            layer_dims[l-1],
            layer_dims[l]
        ) * np.sqrt(2 / layer_dims[l-1])

        # Biases are initialized to zero.
        parameters["b"+str(l)] = np.zeros((1, layer_dims[l]))
    return parameters

In [7]:
# Perform batch-wise forward propagation.
# Every layer computes
# Z = A(previous) × W + b
# A = Activation(Z)
# Hidden Layers → ReLU
# Output Layer → Softmax
def forward_propagation(X, parameters):
    cache = {}
    A = X
    cache["A0"] = X
    L = len(parameters) // 2
    for l in range(1, L):
        # Compute weighted input.
        Z = np.dot(A, parameters["W"+str(l)]) + parameters["b"+str(l)]
        # Apply ReLU activation.
        A = relu(Z)
        # Store intermediate values required for backpropagation.
        cache["Z"+str(l)] = Z
        cache["A"+str(l)] = A
    # Output layer computation.
    Z = np.dot(A, parameters["W"+str(L)]) + parameters["b"+str(L)]
    # Convert logits into probabilities
    A = softmax(Z)
    cache["Z"+str(L)] = Z
    cache["A"+str(L)] = A
    return A, cache

In [8]:
# Perform vectorized backpropagation.
# The gradients are computed using the chain rule
# without automatic differentiation.
def backward_propagation(parameters, cache, X, Y):
    grads = {}
    m = X.shape[0]
    L = len(parameters) // 2
    AL = cache["A"+str(L)]
    # Gradient of Cross-Entropy Loss with Softmax.
    dZ = AL - Y
    for l in reversed(range(1, L + 1)):
        A_prev = cache["A"+str(l - 1)]
        # Gradient of loss with respect to weights.
        grads["dW"+str(l)] = np.dot(A_prev.T, dZ) / m
        # Gradient of loss with respect to biases.
        grads["db"+str(l)] = np.sum(dZ, axis=0, keepdims=True) / m
        if l > 1:
            # Propagate gradient to previous layer.
            dA_prev = np.dot(dZ, parameters["W"+str(l)].T)
            # Apply local Jacobian of ReLU activation.
            dZ = dA_prev * relu_derivative(cache["Z"+str(l-1)])
    return grads

In [9]:
# Update every weight and bias using
# Gradient Descent optimization.
# Parameter = Parameter − Learning Rate × Gradient
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        parameters["W"+str(l)] -= learning_rate * grads["dW"+str(l)]
        parameters["b"+str(l)] -= learning_rate * grads["db"+str(l)]
    return parameters

In [10]:
# Predict the class having the highest probability.
def predict(X, parameters):
    probabilities, _ = forward_propagation(X, parameters)
    return np.argmax(probabilities, axis=1)

In [11]:
# Compute classification accuracy.
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [12]:
# Train the MLP using repeated
# Forward Pass → Loss Calculation →
# Backward Pass → Gradient Descent.
def train(X, y, layer_dims, epochs=1000, learning_rate=0.01):
    parameters = initialize_parameters(layer_dims)
    Y = one_hot(y, layer_dims[-1])
    losses = []
    for epoch in range(epochs):
        output, cache = forward_propagation(X, parameters)
        loss = cross_entropy_loss(Y, output)
        grads = backward_propagation(parameters, cache, X, Y)
        parameters = update_parameters(parameters, grads, learning_rate)
        losses.append(loss)
        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    return parameters, losses